In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, FloatSlider, Text, Checkbox
from IPython.display import display

def solve_intersection(x0, y0, angle_deg, H, ca_rad, top=True):
    """
    Solve intersection of an angled line from (x0,y0) at angle_deg (degrees) with either top or bottom line.
    top_line: y = H - tan(ca_rad)*x
    bottom_line: y = -H + tan(ca_rad)*x
    """
    angle_rad = np.radians(angle_deg)
    dx = np.cos(angle_rad)
    dy = np.sin(angle_rad)
    
    # Choose line equation based on top or bottom
    if top:
        # y_top = H - tan(ca_rad)*x
        slope = -np.tan(ca_rad)
        intercept = H
    else:
        # y_bottom = -H + tan(ca_rad)*x
        slope = np.tan(ca_rad)
        intercept = -H
    
    denom = (dy - slope*dx)
    if abs(denom) < 1e-12:
        return None, None  # No intersection (parallel lines)
    
    rhs = intercept + slope*x0 - y0
    t = rhs/denom
    if t < 0:
        # Intersection behind start point if t<0
        return None, None
    
    x_int = x0 + t*dx
    y_int = y0 + t*dy
    return x_int, y_int

def plot_pattern(num_segments=5, converge_angle=5.0, inner_angle_dev=2.0, inner_line_color='red', 
                 total_width=5.0, total_height=2.0, show_plot=True, save_svg=False, save_filename=''):
    """
    Plot the pattern with:
    - Specified number of segments (num_segments)
    - Converge angle (converge_angle)
    - Inner angle deviation (inner_angle_dev)
    - Inner line color (inner_line_color)
    - Total width and height of the pattern
    - No last horizontal segment
    - First and last angled lines are black
    - Optionally save the figure as an SVG if save_svg and save_filename are set
    """
    H = total_height/2
    ca_rad = np.radians(converge_angle)
    
    x_coords = np.linspace(0, total_width, num_segments+1)
    
    top_angles = []
    bottom_angles = []
    for i in range(num_segments):
        if i%2==0:
            top_angles.append(90 + inner_angle_dev)
            bottom_angles.append(270 - inner_angle_dev)
        else:
            top_angles.append(90 - inner_angle_dev)
            bottom_angles.append(270 + inner_angle_dev)
    
    # Compute intersections for boundary limit
    top_intersections = []
    bottom_intersections = []
    for i in range(num_segments):
        x_i = x_coords[i]
        tx, ty = solve_intersection(x_i, 0, top_angles[i], H, ca_rad, top=True)
        bx, by = solve_intersection(x_i, 0, bottom_angles[i], H, ca_rad, top=False)
        if tx is not None and bx is not None:
            top_intersections.append((tx, ty))
            bottom_intersections.append((bx, by))
    
    fig, ax = plt.subplots(figsize=(8,8))
    
    if len(top_intersections)==0 or len(bottom_intersections)==0:
        # If no intersections, just show empty if show_plot
        if show_plot:
            plt.show()
        # Optionally save empty figure if requested
        if save_svg and save_filename.strip():
            fig.savefig(save_filename.strip(), format='svg')
        return
    
    # Determine x-limits from intersections
    all_x = [p[0] for p in top_intersections] + [p[0] for p in bottom_intersections]
    x_min = min(all_x)
    x_max = max(all_x)
    
    # Sample points on top/bottom lines
    x_line = np.linspace(x_min, x_max, 100)
    y_top_line = H - np.tan(ca_rad)*x_line
    y_bottom_line = -H + np.tan(ca_rad)*x_line
    
    # Draw top and bottom lines within these limits
    ax.plot(x_line, y_top_line, color='black', lw=2)
    ax.plot(x_line, y_bottom_line, color='black', lw=2)
    
    # Middle horizontal line y=0: draw one fewer than num_segments lines
    for i in range(num_segments-1):
        color = 'blue' if i%2==0 else 'red'
        ax.plot([x_coords[i], x_coords[i+1]], [0,0], color=color, lw=2)
    
    # Draw inner angled segments:
    # If i=0 or i=num_segments-1, angled lines are black
    # Otherwise use inner_line_color
    for i in range(num_segments):
        x_i = x_coords[i]
        tx, ty = solve_intersection(x_i, 0, top_angles[i], H, ca_rad, top=True)
        bx, by = solve_intersection(x_i, 0, bottom_angles[i], H, ca_rad, top=False)
        
        if tx is None or bx is None:
            continue
        
        # Determine line color
        line_col = 'black' if (i==0 or i==num_segments-1) else inner_line_color
        
        # Top angled line
        ax.plot([x_i, tx], [0, ty], color=line_col, lw=2)
        # Bottom angled line
        ax.plot([x_i, bx], [0, by], color=line_col, lw=2)
        # No dotted line connecting top and bottom intersections as requested
    
    ax.set_aspect('equal', 'box')
    y_all = [p[1] for p in top_intersections] + [p[1] for p in bottom_intersections] + [0]
    y_min = min(y_all) - 0.5
    y_max = max(y_all) + 0.5
    ax.set_xlim(x_min-0.5, x_max+0.5)
    ax.set_ylim(y_min, y_max)
    ax.axis('off')
    
    if show_plot:
        plt.show()
    if save_svg and save_filename.strip():
        fig.savefig(save_filename.strip(), format='svg')

def interactive_pattern():
    widget = interactive(
        plot_pattern,
        num_segments=IntSlider(min=1, max=20, step=1, value=5, description='Segments'),
        converge_angle=FloatSlider(min=0.0, max=45.0, step=1.0, value=5.0, description='Converge Angle (deg)'),
        inner_angle_dev=FloatSlider(min=-45.0, max=45.0, step=1.0, value=2.0, description='Inner Angle Dev (deg)'),
        inner_line_color=Text(value='red', description='Inner Line Color'),
        total_width=FloatSlider(min=1.0, max=20.0, step=1.0, value=15.0, description='Total Width'),
        total_height=FloatSlider(min=1.0, max=20.0, step=1.0, value=3.0, description='Total Height'),
        show_plot=Checkbox(value=True, description='Show Plot'),
        save_svg=Checkbox(value=False, description='Save SVG'),
        save_filename=Text(value='', description='SVG Filename')
    )
    display(widget)

# Display the interactive widget
interactive_pattern()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


interactive(children=(IntSlider(value=5, description='Segments', max=20, min=1), FloatSlider(value=5.0, descri…